
# From RNNs to Transformers

## Why This Matters

Many researchers work with sequence data:

-   clinical notes and patient timelines
-   instrument logs and time-ordered measurements
-   reports, interviews, and long text documents

This chapter is for conceptual understanding, not deep model implementation.

Guiding question: **How can a model decide which parts of a long sequence are relevant at a specific position?**

## Running Example

Consider this clinical sentence:

*"Patient admitted three days ago with acute chest pain, treated with aspirin, now reports reduced discomfort."*

When interpreting "discomfort", the model should connect to earlier words like "chest pain" and "aspirin". The challenge is retrieving useful context across long distances.

## RNNs: First Main Solution

An RNN reads tokens one by one and keeps a hidden state (memory): $$h_t = \tanh(Wx_t + Uh_{t-1})$$

Interpretation:

-   `x_t`: current token representation
-   `h_{t-1}`: previous memory
-   `h_t`: updated memory

RNN strengths:

-   natural for ordered/time-dependent data
-   simple repeated update rule

RNN limitations:

1.  **Sequential bottleneck**: token 100 waits for token 99.
2.  **Memory bottleneck**: early details can fade in long sequences.

**LSTMs** (Long Short-Term Memory networks) improved on this with gates that control what to remember and forget. They were the dominant sequence model from roughly 2014&#x2013;2017, but remained sequential and still struggled with very long contexts.

### Demo: Sequential Processing in an RNN

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

rnn_cell = nn.RNNCell(input_size=4, hidden_size=4)
words = ["Patient", "reports", "chest", "pain"]
embeddings = [torch.randn(1, 4) for _ in words]

h = torch.zeros(1, 4)
for i, (word, emb) in enumerate(zip(words, embeddings), start=1):
    h = rnn_cell(emb, h)
    print(f"Step {i}: {word:>8} -> h[:2] = {h[0, :2].tolist()}")

Step 1:  Patient -> h[:2] = [0.9314628839492798, 0.617828369140625]
Step 2:  reports -> h[:2] = [0.6935661435127258, 0.14799222350120544]
Step 3:    chest -> h[:2] = [0.8815560936927795, 0.5947684645652771]
Step 4:     pain -> h[:2] = [0.7856016755104065, 0.34498998522758484]


**Takeaway**: each step depends on the previous one. This gives sequence awareness but also limits speed and long-context reliability.

## Why Attention Was Introduced

RNNs must compress the entire past into one vector. Attention changes this:

Instead of "remember everything in one state", the model does a dynamic lookup: **Which earlier tokens matter most right now?**

Attention weights:

-   are non-negative
-   sum to 1
-   are different for each token/query

### Intuition Demo: Weighted Lookup

In [2]:
import numpy as np

tokens = ["Patient", "reports", "chest", "pain", "today"]

# Illustrative weights for interpreting "today"
w = np.array([0.05, 0.25, 0.35, 0.30, 0.05])
w = w / w.sum()

print("Attention weights:")
for tok, weight in zip(tokens, w):
    print(f"  {tok:>8}: {weight:.2f}")
print(f"       sum: {w.sum():.2f}")

Attention weights:
   Patient: 0.05
   reports: 0.25
     chest: 0.35
      pain: 0.30
     today: 0.05
       sum: 1.00


**Takeaway**: attention is token-specific relevance weighting, not a fixed memory vector.

## Transformers: Attention-Centered Architecture

Transformers (Vaswani et al., 2017) remove recurrence and build around attention.

Practical difference:

| Feature       | RNN / LSTM                     | Transformer                                       |
|------------- |------------------------------ |------------------------------------------------- |
| Processing    | Sequential                     | Parallel in training                              |
| Context       | Compressed in one hidden state | Direct weighted access across positions           |
| Long distance | Degrades with sequence length  | One attention step away (subject to masking rule) |

### One Core Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Plain-language mapping:

-   **Q (Query)**: what this token is looking for
-   **K (Key)**: what each token can offer
-   **V (Value)**: the actual information each token holds

Attention works like a soft dictionary lookup:

1.  Compare each Query against all Keys (dot product) to get relevance scores.
2.  Normalize the scores with softmax to get weights that sum to 1.
3.  Use the weights to take a weighted sum of Values &#x2014; retrieving a blend of information from the most relevant positions.

The $\sqrt{d_k}$ scaling prevents dot products from growing too large, which would push softmax into extreme regions where gradients vanish.

### Transformer Block (Conceptual)

Each block combines:

1.  **Self-attention** (aggregate information across positions, as described above)
2.  **Feed-forward network** (a small MLP applied independently at each position, adding non-linearity and capacity)
3.  **Residual connections** (the input to each sub-layer is added back to its output, helping gradients flow through deep networks)
4.  **Layer normalization** (stabilizes training by normalizing activations within each layer)
5.  **Positional encoding** (injects token order, since attention itself treats input as a set &#x2014; without it, "Dog bites Man" = "Man bites Dog")

### Multi-Head Attention

Rather than computing a single set of attention weights, transformers run multiple attention operations in parallel &#x2014; called **heads** &#x2014; each with its own Q, K, V projections. Different heads can learn to focus on different aspects: one might track syntactic relationships (subject-verb agreement), another semantic similarity, another local proximity. The outputs of all heads are concatenated and projected back to the model dimension, giving the model a richer, multi-faceted view of token relationships.

### Model Families

-   **Encoder-only** (BERT): bidirectional attention over input, suited for classification and understanding tasks
-   **Decoder-only** (GPT): causal masking (past-only attention), suited for text generation
-   **Encoder-decoder** (T5): combines both, used in translation and summarization tasks

## Why GPT Generates Left-to-Right

GPT uses **causal masking**:

-   token $t$ can attend only to tokens $\leq t$
-   future positions are blocked

So:

-   training can still be parallel across positions
-   generation remains token-by-token

### Demo: Causal Masking

In [3]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
n = 5
scores = torch.randn(n, n)

# Upper triangle marks future tokens
mask = torch.triu(torch.ones(n, n), diagonal=1)
masked_scores = scores.masked_fill(mask == 1, float('-inf'))
weights = F.softmax(masked_scores, dim=-1)

print("Row 0:", [f"{v:.2f}" for v in weights[0].tolist()])
print("Row 3:", [f"{v:.2f}" for v in weights[3].tolist()])

Row 0: ['1.00', '0.00', '0.00', '0.00', '0.00']
Row 3: ['0.20', '0.06', '0.17', '0.56', '0.00']


**Takeaway**: early rows have less context; later rows can attend to more past tokens.

## Practical View for Domain Researchers

You typically do **not** train transformers from scratch.

Common workflow:

1.  choose a pretrained model appropriate for your domain and task
2.  start with prompting and retrieval
3.  evaluate on your domain-specific examples
4.  fine-tune only if necessary
5.  keep expert review for high-stakes decisions

For example, a geologist classifying rock descriptions does not need to train a transformer from scratch &#x2014; they can use a pretrained model and fine-tune on a few hundred labelled examples.

## Exercises

### RNN Sensitivity to Early vs Late Tokens

Goal: compare how perturbing early vs late tokens changes final hidden state.

In [4]:
import torch
import torch.nn as nn

torch.manual_seed(42)
rnn_cell = nn.RNNCell(input_size=4, hidden_size=4)
seq = [torch.randn(1, 4) for _ in range(10)]

def run(sequence):
    h = torch.zeros(1, 4)
    for emb in sequence:
        h = rnn_cell(emb, h)
    return h

h_base = run(seq)

seq_early = [x.clone() for x in seq]
seq_early[0] = seq_early[0] + 2.0
h_early = run(seq_early)

seq_late = [x.clone() for x in seq]
seq_late[-1] = seq_late[-1] + 2.0
h_late = run(seq_late)

print("L2 diff (early change):", torch.norm(h_base - h_early).item())
print("L2 diff (late change): ", torch.norm(h_base - h_late).item())

L2 diff (early change): 0.0001328006328549236
L2 diff (late change):  1.4259880781173706


**Questions to consider**:

-   Which perturbation (early or late) produces a larger difference? Why?
-   What would happen if you increased the sequence length to 30? Would the gap between early and late grow or shrink?

### Build Attention Weights

Goal: compute a self-attention matrix and verify row-wise probability behavior.

In [5]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
x = torch.randn(1, 5, 8)
scores = torch.matmul(x, x.transpose(-2, -1)) / (8 ** 0.5)
weights = F.softmax(scores, dim=-1)

print("Weights shape:", weights.shape)
print("Row sums:", weights[0].sum(dim=-1))
print("Unmasked row 0:", [f"{v:.2f}" for v in weights[0, 0].tolist()])

Weights shape: torch.Size([1, 5, 5])
Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
Unmasked row 0: ['0.97', '0.01', '0.00', '0.02', '0.00']


**Questions to consider**:

-   Each row is one token's attention distribution. Which row attends most uniformly across all tokens? Which is most concentrated on a single token?
-   What does it mean when a token assigns high weight to itself?

### Compare Unmasked vs Masked Attention

Goal: see how causal masking changes attention for early and late positions.

In [6]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
x = torch.randn(1, 5, 8)
scores = torch.matmul(x, x.transpose(-2, -1)) / (8 ** 0.5)

unmasked = F.softmax(scores, dim=-1)

mask = torch.triu(torch.ones(5, 5), diagonal=1)
masked_scores = scores.masked_fill(mask == 1, float('-inf'))
masked = F.softmax(masked_scores, dim=-1)

print("Unmasked row 0:", [f"{v:.2f}" for v in unmasked[0, 0].tolist()])
print("Masked   row 0:", [f"{v:.2f}" for v in masked[0, 0].tolist()])
print("Unmasked row 2:", [f"{v:.2f}" for v in unmasked[0, 2].tolist()])
print("Masked   row 2:", [f"{v:.2f}" for v in masked[0, 2].tolist()])
print("Unmasked row 4:", [f"{v:.2f}" for v in unmasked[0, 4].tolist()])
print("Masked   row 4:", [f"{v:.2f}" for v in masked[0, 4].tolist()])

Unmasked row 0: ['0.97', '0.01', '0.00', '0.02', '0.00']
Masked   row 0: ['1.00', '0.00', '0.00', '0.00', '0.00']
Unmasked row 2: ['0.02', '0.03', '0.93', '0.02', '0.01']
Masked   row 2: ['0.02', '0.03', '0.95', '0.00', '0.00']
Unmasked row 4: ['0.00', '0.01', '0.00', '0.02', '0.97']
Masked   row 4: ['0.00', '0.01', '0.00', '0.02', '0.97']


**Questions to consider**:

-   Row 0 in the masked version can only attend to token 0. What does this mean for the first token's representation compared to later tokens?
-   How do row 4's masked weights compare to its unmasked weights? Where did the "lost" weight from future tokens go?

## Summary

1.  **RNNs**: sequence models with sequential memory updates.
2.  **Attention**: weighted lookup over relevant positions.
3.  **Transformers**: attention-first models with better long-context scaling.
4.  **GPT-style LLMs**: decoder transformers with causal masking.
5.  **Applied use**: pretrained-first workflow + domain evaluation + expert oversight.